<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/Anh/src/notebooks/Simple_interpretable_baseline_Seniority%26Domain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Simple Interpretable Baseline - Seniority and Domain Prediction**
In this approach, we model seniority and domain prediction as a text classification problem using job titles and descriptions. We apply Bag-of-Words and TF–IDF representations to transform raw text into numerical features, followed by standard classifiers such as Logistic Regression, Random Forest, and CatBoost. This approach serves as a strong and interpretable baseline, allowing us to evaluate how much signal is already contained in textual job information without heavy feature engineering.

### Preparing the dataset

In [1]:
!pip install catboost
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.5 MB/s eta 0:00:00


In [2]:
with open("/content/linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []
for cv in cvs:
    for job in cv:
        jobs.append(job)

df = pd.DataFrame(jobs)
df_active = df[df["status"] == "ACTIVE"]
df_active.head()


,organization,linkedin,position,startDate,endDate,status,department,seniority
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management


## Spliting the dataset and encoding target

In [3]:
# Seniority
## Spliting the data
X_sn = df_active["position"]
y_sn = df_active["seniority"]
X_train_sn, X_test_sn, y_train_sn, y_test_sn = train_test_split(
    X_sn, y_sn, test_size=0.2, random_state=42, stratify=y_sn)

## Encode y
le = LabelEncoder()
y_train_enc_sn = le.fit_transform(y_train_sn)
y_test_enc_sn = le.transform(y_test_sn)

In [4]:
# Domain
## Spliting Data
X_dm = df_active["position"]
y_dm = df_active["department"]
X_train_dm, X_test_dm, y_train_dm, y_test_dm = train_test_split(
    X_dm, y_dm, test_size=0.2, random_state=42, stratify=y_dm)

## Encode y
y_train_enc_dm = le.fit_transform(y_train_dm)
y_test_enc_dm = le.transform(y_test_dm)

## Training the model

Three models were selected for training: Logistic Regression, Random Forest, and CatBoost.
- Logistic Regression was chosen as a strong linear baseline that is suited for high dimensional and sparse features generated by BoW and TF–IDF.
- Random Forest was included to capture nonlinear relationships and interactions between features that linear models may miss.
- CatBoost was selected as a more advanced ensemble method due to its robustness and strong performance in structured prediction tasks, providing a competitive benchmark against the simpler models.

In [5]:
## define model
random_forest = RandomForestClassifier(n_estimators=200, random_state=42)
CatBoost = CatBoostClassifier(verbose=False,random_seed=42)
Logistic_Regression = LogisticRegression(max_iter=1000)

In this approach, two text representation methods were selected: Bag-of-Words (BoW) and TF–IDF.

- BoW was a simple and interpretable baseline, where text is represented by word occurrence frequencies. This allows the model to capture common patterns in job titles without introducing additional weighting assumptions.
- TF–IDF was then applied to improve upon BoW by down weighting frequently occurring but less informative words, while emphasizing terms that are more distinctive for specific domains or seniority levels.

Using both representations enables a comparison between a basic frequency based method and a more informative text encoding.

In [6]:
# Senniority

## Train pipeline Bag-of-Words
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("bow", CountVectorizer(lowercase=True, stop_words='english')),
         ("model", model)])
    pipeline.fit(X_train_sn, y_train_enc_sn)
    y_pred_sn_bow = pipeline.predict(X_test_sn)
    acc = accuracy_score(y_test_enc_sn, y_pred_sn_bow)
    print(f"{type(model).__name__} Accuracy (BoW): {acc:.3f}")

## Train pipeline with TF–IDF
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("tfidf",TfidfVectorizer(lowercase=True, stop_words='english')),
        ("model", model)])
    pipeline.fit(X_train_sn, y_train_enc_sn)
    y_pred_sn = pipeline.predict(X_test_sn)
    acc_tf = accuracy_score(y_test_enc_sn, y_pred_sn)
    print(f"{type(model).__name__} Accuracy (TF–IDF): {acc_tf:.3f}")

RandomForestClassifier Accuracy (BoW): 0.816
CatBoostClassifier Accuracy (BoW): 0.808
LogisticRegression Accuracy (BoW): 0.792
RandomForestClassifier Accuracy (TF–IDF): 0.808
CatBoostClassifier Accuracy (TF–IDF): 0.784
LogisticRegression Accuracy (TF–IDF): 0.760


Overall, BoW consistently outperforms TF–IDF across all three models for seniority prediction. The best performance is achieved by Random Forest with BoW withAccuracy = 0.816, followed closely by CatBoost. This suggests that absolute word frequency in job titles carries strong signals for seniority levels, and additional TF–IDF weighting does not provide extra benefit. Logistic Regression performs reasonably well but lags behind tree-based models, indicating that non-linear decision boundaries help capture seniority patterns more effectively.

In [7]:
# Domain

## Train pipeline Bag-of-Words
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("bow", CountVectorizer(lowercase=True, stop_words='english')),
        ("model", model)])
    pipeline.fit(X_train_dm, y_train_enc_dm)
    y_pred_dm_bow = pipeline.predict(X_test_dm)
    acc_dm_bow = accuracy_score(y_test_enc_dm, y_pred_dm_bow)
    print(f"{type(model).__name__} Accuracy (BoW): {acc_dm_bow:.3f}")

## Train pipeline with TF–IDF
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("tfidf",TfidfVectorizer(lowercase=True, stop_words='english')),
        ("model", model)])
    pipeline.fit(X_train_dm, y_train_enc_dm)
    y_pred_dm_tf = pipeline.predict(X_test_dm)
    acc_dm_tf = accuracy_score(y_test_enc_dm, y_pred_dm_tf)
    print(f"{type(model).__name__} Accuracy (TF–IDF): {acc_dm_tf:.3f}")

RandomForestClassifier Accuracy (BoW): 0.768
CatBoostClassifier Accuracy (BoW): 0.736
LogisticRegression Accuracy (BoW): 0.720
RandomForestClassifier Accuracy (TF–IDF): 0.776
CatBoostClassifier Accuracy (TF–IDF): 0.720
LogisticRegression Accuracy (TF–IDF): 0.632


For domain classification, performance differences between BoW and TF–IDF are smaller. Random Forest with TF–IDF achieves the highest accuracy (0.776), slightly outperforming BoW. This indicates that domain-specific keywords benefit more from TF–IDF weighting, where rare but informative terms play a stronger role. Logistic Regression shows a noticeable performance drop with TF–IDF, suggesting it is more sensitive to feature scaling and sparsity compared to ensemble models.

The results show that feature representation and model choice should be task-dependent.
For seniority prediction, BoW combined with tree-based models performs best, as seniority-related cues are often explicit and frequency-based. In contrast, domain prediction benefits more from TF–IDF, where distinguishing keywords are more informative. Across both tasks, Random Forest demonstrates the most stable and robust performance, making it a strong overall choice for text-based classification in this setting.

#